# Test Bronze — INE
Verifica las 5 tablas Bronze de INE (AWS S3).  
Usa `limit(10)` — no ejecuta full scan.

In [0]:
from pyspark.sql import functions as F

BRONZE_SCHEMA = "sandbox_bronze_ss2"

_EDAD_TABLES = [
    "ine_defunciones_sexo_edad_causas_muerte",
    "ine_defunciones_neonatales_sexo_edad_causas_muerte",
    "ine_defunciones_postneonatales_sexo_edad_causas_muerte",
]
_GEO_TABLES = [
    "ine_defunciones_sexo_depto_residencia_causas_muerte",
    "ine_defunciones_causas_externas_sexo_depto_ocurrencia",
]
_ALL_TABLES = _EDAD_TABLES + _GEO_TABLES

# Columnas de auditoría presentes en todas las tablas INE
_AUDIT_COLS = ["ingestion_timestamp", "source_system", "source_file", "s3_key", "excel_sheet"]

---
## Tablas INE — Edad

In [0]:
df = spark.table(f"{BRONZE_SCHEMA}.ine_defunciones_sexo_edad_causas_muerte")
df.printSchema()

In [0]:
display(df.limit(10))

In [0]:
df = spark.table(f"{BRONZE_SCHEMA}.ine_defunciones_neonatales_sexo_edad_causas_muerte")
df.printSchema()

In [0]:
display(df.limit(10))

In [0]:
df = spark.table(f"{BRONZE_SCHEMA}.ine_defunciones_postneonatales_sexo_edad_causas_muerte")
df.printSchema()

In [0]:
display(df.limit(10))

---
## Tablas INE — Geografía

In [0]:
df = spark.table(f"{BRONZE_SCHEMA}.ine_defunciones_sexo_depto_residencia_causas_muerte")
df.printSchema()

In [0]:
display(df.limit(10))

In [0]:
df = spark.table(f"{BRONZE_SCHEMA}.ine_defunciones_causas_externas_sexo_depto_ocurrencia")
df.printSchema()

In [0]:
display(df.limit(10))

---
## Chequeos de calidad (sin full scan)

In [0]:
# 1. Columnas de auditoría presentes
print("── Columnas de auditoría Bronze presentes ──")
for t in _ALL_TABLES:
    cols = spark.table(f"{BRONZE_SCHEMA}.{t}").columns
    missing = [c for c in _AUDIT_COLS if c not in cols]
    print(f"  {t}: {'OK' if not missing else f'WARN — faltan: {missing}'}")

In [0]:
# 2. Nulls en columnas clave (muestra 1k)
print("── Nulls en columnas clave (muestra 1k) ──")
_KEY_COLS = ["anio", "total", "hombres", "mujeres"]
for t in _ALL_TABLES:
    sample = spark.table(f"{BRONZE_SCHEMA}.{t}").limit(1000)
    cols_presentes = [c for c in _KEY_COLS if c in sample.columns]
    print(f"\n  {t}")
    for col in cols_presentes:
        nulls = sample.filter(F.col(col).isNull()).count()
        print(f"    {col} nulls: {nulls}")

In [0]:
# 3. Auditoría sin nulls
print("── Nulls en columnas de auditoría (muestra 1k) ──")
for t in _ALL_TABLES:
    sample = spark.table(f"{BRONZE_SCHEMA}.{t}").limit(1000)
    cols_presentes = [c for c in _AUDIT_COLS if c in sample.columns]
    print(f"\n  {t}")
    for col in cols_presentes:
        nulls = sample.filter(F.col(col).isNull()).count()
        status = "OK" if nulls == 0 else f"WARN — {nulls} nulls"
        print(f"    {col}: {status}")

In [0]:
# 4. Rango de años
print("── Rango de anio (muestra 5k) ──")
for t in _ALL_TABLES:
    result = (
        spark.table(f"{BRONZE_SCHEMA}.{t}")
        .limit(5000)
        .agg(F.min("anio").alias("min"), F.max("anio").alias("max"))
        .collect()[0]
    )
    print(f"  {t}: {result['min']} — {result['max']}")